# Демонстрация Tiny U-Net / Tiny U-Net demonstration
**RU.** Дополнительная демонстрация к лекции 12, не пятое задание практики. CPU; требуется PyTorch. Используются те же реальные изображения и фиксированное разбиение. Это маленькая одноуровневая сеть с skip connection, не точная оригинальная U-Net. 30 эпох, Adam 0.003, порог 0.5 фиксированы до test. Показанный на лекции запуск предсказал фон и получил test IoU=0: это диагностический пример, не оценка потенциала архитектуры. Не меняйте параметры по результату test; новые гипотезы проверяйте в новом протоколе на train/dev.
**EN.** Optional lecture demonstration, not a fifth practice exercise. CPU; requires PyTorch. Uses the same real images and fixed partitions. This tiny one-level skip-connected network is not the original U-Net. Thirty epochs, Adam 0.003, and threshold 0.5 are fixed before test. The lecture run predicted background and obtained test IoU=0: a diagnostic case, not an assessment of the architecture's potential. Do not tune using test feedback; investigate new hypotheses under a new train/dev protocol.


# 12 · Сегментация / Image segmentation
**RU.** Реальные фотографии Oxford-IIIT Pet (Parkhi et al., 2012): наблюдение — изображение животного, признаки — RGB-пиксели, цель — маска foreground/background. [Официальный источник](https://www.robots.ox.ac.uk/~vgg/data/pets/). Trimap: 1 — объект, 2 — фон, 3 — неоднозначная граница; значение 3 исключаем из метрик. Это учебный бинарный протокол, не классификация пород и не официальная benchmark-оценка. Данные доступны для исследований; соблюдайте условия исходных сайтов изображений, указанные авторами. Изображения не распространяются в репозитории курса.
**EN.** Real Oxford-IIIT Pet photographs: each observation is an animal image, RGB pixels are features, and a foreground/background mask is the target. Official trimaps use 1 for foreground, 2 for background, 3 for unclassified boundary. Exclude label 3 from metrics. This binary teaching protocol is neither breed classification nor an official benchmark score. The dataset is released for research; original image website terms still apply. Images are not redistributed in this repository.

**RU.** CPU; NumPy, Pillow, OpenCV, matplotlib. Первый запуск скачивает около 800 МБ, повторный использует кэш. Зафиксированы 40 train, 8 dev и 8 test: равное число кошек и собак, seed 1212. Train/dev — из официального trainval, test — из test. Идентификаторы непересекаются; сведения об отдельных животных отсутствуют, независимость по животным не заявляется. Маски не используются для выбора рамки. Параметры выбираем только по dev; test открываем после фиксации правила.
**EN.** CPU; NumPy, Pillow, OpenCV, matplotlib. Initial download is about 800 MB; subsequent runs reuse the cache. Fixed 40 train, 8 dev and 8 test images, equally many cats and dogs, seed 1212. Train/dev come from official trainval; test comes from official test. IDs are disjoint, but animal identity independence is not established. Masks do not determine input rectangles. Tune on dev, freeze the rule, then evaluate test.


In [ ]:
from pathlib import Path  # RU: управляем кэшем; EN: manage the cache.
import json, hashlib, tarfile, urllib.request  # RU: читаем и проверяем архивы; EN: read and verify archives.
import numpy as np  # RU: обрабатываем маски; EN: process masks.
from PIL import Image  # RU: декодируем изображения; EN: decode images.
import cv2  # RU: используем GrabCut; EN: use GrabCut.
import matplotlib.pyplot as plt  # RU: показываем фотографии; EN: display photographs.
data = next((p for p in [Path('build/data'), Path('../build/data'), Path('data')] if (p/'images.tar.gz').exists()), Path('data'))  # RU: находим или создаём кэш; EN: locate or create the cache.
data.mkdir(exist_ok=True, parents=True)  # RU: создаём каталог данных; EN: create the data directory.
resources = {'images.tar.gz':'5c4f3ee8e5d25df40f4fd59a7f44e54c', 'annotations.tar.gz':'95a8c909bbe2e81eed6a22bccdf3f68f'}  # RU: фиксируем контрольные суммы torchvision; EN: pin torchvision checksums.
for filename, checksum in resources.items():  # RU: проверяем оба архива; EN: check both archives.
    path = data/filename  # RU: формируем путь архива; EN: construct archive path.
    if not path.exists():  # RU: избегаем повторной загрузки; EN: avoid repeat downloads.
        urllib.request.urlretrieve('https://www.robots.ox.ac.uk/~vgg/data/pets/data/'+filename, path)  # RU: загружаем официальный архив; EN: download the official archive.
    assert hashlib.md5(path.read_bytes()).hexdigest() == checksum  # RU: проверяем целостность; EN: verify integrity.
PARTS = {'train': ['Egyptian_Mau_155', 'Bombay_175', 'Egyptian_Mau_190', 'Russian_Blue_150', 'Abyssinian_1', 'Maine_Coon_141', 'Bengal_183', 'Egyptian_Mau_132', 'Abyssinian_114', 'Sphynx_200', 'Ragdoll_159', 'Abyssinian_197', 'Ragdoll_125', 'Egyptian_Mau_147', 'Siamese_155', 'Sphynx_115', 'Maine_Coon_150', 'Bombay_107', 'Bengal_136', 'Birman_155', 'german_shorthaired_180', 'keeshond_184', 'english_cocker_spaniel_103', 'staffordshire_bull_terrier_131', 'shiba_inu_106', 'great_pyrenees_142', 'chihuahua_136', 'german_shorthaired_181', 'german_shorthaired_113', 'wheaten_terrier_112', 'staffordshire_bull_terrier_159', 'chihuahua_187', 'yorkshire_terrier_186', 'yorkshire_terrier_118', 'japanese_chin_108', 'basset_hound_179', 'japanese_chin_120', 'wheaten_terrier_142', 'keeshond_163', 'german_shorthaired_13'], 'dev': ['British_Shorthair_182', 'Bengal_104', 'Egyptian_Mau_107', 'Ragdoll_143', 'shiba_inu_142', 'keeshond_158', 'english_setter_151', 'basset_hound_147'], 'test': ['Egyptian_Mau_213', 'Abyssinian_43', 'Russian_Blue_207', 'Maine_Coon_268', 'wheaten_terrier_69', 'staffordshire_bull_terrier_40', 'chihuahua_191', 'beagle_61']}  # RU: сохраняем идентификаторы разбиения; EN: preserve split IDs.
assert not (set(PARTS['train']) & set(PARTS['dev']) or set(PARTS['train']) & set(PARTS['test']) or set(PARTS['dev']) & set(PARTS['test']))  # RU: исключаем пересечения; EN: reject overlapping IDs.
images, trimaps = {}, {}  # RU: создаём коллекции наблюдений и меток; EN: create observation and label collections.
with tarfile.open(data/'images.tar.gz') as archive:  # RU: читаем без распаковки путей; EN: read without extracting paths.
    for name in sum(PARTS.values(), []):  # RU: загружаем фиксированную подвыборку; EN: load the fixed subset.
        images[name] = np.asarray(Image.open(archive.extractfile('images/'+name+'.jpg')).convert('RGB').resize((128,128), Image.Resampling.BILINEAR))  # RU: масштабируем RGB; EN: resize RGB images.
with tarfile.open(data/'annotations.tar.gz') as archive:  # RU: открываем разметку; EN: open annotations.
    for name in images:  # RU: сопоставляем по имени, не позиции; EN: match by ID rather than position.
        trimaps[name] = np.asarray(Image.open(archive.extractfile('annotations/trimaps/'+name+'.png')).resize((128,128), Image.Resampling.NEAREST))  # RU: сохраняем дискретные метки; EN: preserve discrete labels.
name = PARTS['dev'][0]  # RU: выбираем пример dev; EN: select a development example.
fig, axes = plt.subplots(1,2,figsize=(8,4))  # RU: сопоставляем изображение и разметку; EN: compare image and annotation.
axes[0].imshow(images[name]); axes[1].imshow(trimaps[name],vmin=1,vmax=3)  # RU: показываем исходные данные; EN: display source data.
plt.show()  # RU: выводим рисунок; EN: render the figure.


In [ ]:
import torch  # RU: строим небольшую сеть; EN: build a small network.
torch.manual_seed(1212)  # RU: фиксируем инициализацию; EN: fix initialization.
torch.set_num_threads(2)  # RU: ограничиваем CPU-параллелизм; EN: limit CPU parallelism.
class TinyUNet(torch.nn.Module):  # RU: задаём одноуровневую encoder-decoder сеть; EN: define a one-level encoder-decoder.
    def __init__(self):  # RU: создаём обучаемые слои; EN: construct trainable layers.
        super().__init__()  # RU: инициализируем базовый модуль; EN: initialize the base module.
        self.enc = torch.nn.Sequential(torch.nn.Conv2d(3,8,3,padding=1),torch.nn.ReLU(),torch.nn.Conv2d(8,8,3,padding=1),torch.nn.ReLU())  # RU: извлекаем восемь карт признаков; EN: extract eight feature maps.
        self.pool = torch.nn.MaxPool2d(2)  # RU: уменьшаем сетку вдвое; EN: halve spatial resolution.
        self.low = torch.nn.Sequential(torch.nn.Conv2d(8,16,3,padding=1),torch.nn.ReLU())  # RU: получаем шестнадцать контекстных карт; EN: produce sixteen contextual maps.
        self.dec = torch.nn.Sequential(torch.nn.Conv2d(24,8,3,padding=1),torch.nn.ReLU(),torch.nn.Conv2d(8,1,1))  # RU: объединяем 8+16 каналов и выдаём логит; EN: combine 8+16 channels and output a logit.
    def forward(self,x):  # RU: определяем прямой проход; EN: define the forward pass.
        skip = self.enc(x)  # RU: сохраняем детали раннего уровня; EN: retain early spatial detail.
        low = self.low(self.pool(skip))  # RU: вычисляем признаки на меньшей сетке; EN: compute lower-resolution features.
        up = torch.nn.functional.interpolate(low,size=skip.shape[-2:],mode='bilinear',align_corners=False)  # RU: согласуем размеры ветвей; EN: align branch dimensions.
        return self.dec(torch.cat([skip,up],dim=1))[:,0]  # RU: возвращаем карту логитов N×H×W; EN: return N×H×W logits.
def batch(names):  # RU: формируем тензоры настоящих изображений; EN: form tensors from real images.
    x = torch.tensor(np.stack([cv2.resize(images[n],(64,64)) for n in names]).transpose(0,3,1,2)/255,dtype=torch.float32)  # RU: переводим RGB в NCHW и диапазон 0–1; EN: convert RGB to NCHW and range 0–1.
    m = np.stack([cv2.resize(trimaps[n],(64,64),interpolation=cv2.INTER_NEAREST) for n in names])  # RU: масштабируем категориальные метки; EN: resize categorical labels.
    return x,torch.tensor(m==1,dtype=torch.float32),torch.tensor(m!=3,dtype=torch.float32)  # RU: возвращаем вход, цель и valid; EN: return input, target, and validity.
model = TinyUNet()  # RU: создаём модель с фиксированным seed; EN: initialize the seeded model.
optimizer = torch.optim.Adam(model.parameters(),lr=.003)  # RU: фиксируем оптимизатор до test; EN: fix the optimizer before test.
x,y,v = batch(PARTS['train'])  # RU: используем только train для градиентов; EN: use training data for gradients only.
history = []  # RU: записываем функцию потерь; EN: record training loss.
for epoch in range(30):  # RU: выполняем заранее заданный бюджет; EN: run the predeclared budget.
    model.train(); optimizer.zero_grad()  # RU: начинаем шаг обучения; EN: begin a training step.
    logits = model(x)  # RU: вычисляем попиксельные логиты; EN: compute pixelwise logits.
    loss_map = torch.nn.functional.binary_cross_entropy_with_logits(logits,y,reduction='none')  # RU: сохраняем карту потерь; EN: retain the loss map.
    loss = (loss_map*v).sum()/v.sum()  # RU: исключаем неоднозначную границу; EN: exclude ambiguous boundaries.
    loss.backward(); optimizer.step()  # RU: обновляем параметры по train; EN: update parameters using train.
    history.append(float(loss.detach()))  # RU: сохраняем число без графа; EN: store a scalar without the graph.
model.eval()  # RU: фиксируем режим итоговой оценки; EN: freeze evaluation mode.
with torch.no_grad():  # RU: отключаем градиенты на test; EN: disable test gradients.
    test_x,_,_ = batch(PARTS['test'])  # RU: открываем итоговую подвыборку; EN: access the final subset.
    probabilities = torch.sigmoid(model(test_x)).numpy()  # RU: получаем вероятности foreground; EN: obtain foreground probabilities.
scores = []  # RU: сохраняем IoU каждого изображения; EN: retain image-level IoU.
for index,name in enumerate(PARTS['test']):  # RU: проверяем каждое test-наблюдение; EN: evaluate every test observation.
    prediction = cv2.resize(probabilities[index],(128,128),interpolation=cv2.INTER_LINEAR)>=.5  # RU: применяем заранее фиксированный порог; EN: apply the predeclared threshold.
    target,valid = trimaps[name]==1,trimaps[name]!=3  # RU: декодируем политику оценки; EN: decode evaluation policy.
    intersection = np.sum(prediction & target & valid)  # RU: считаем верный foreground; EN: count correct foreground.
    union = np.sum((prediction | target) & valid)  # RU: считаем объединение; EN: count the union.
    scores.append(float(intersection/union) if union else np.nan)  # RU: не выдумываем пустые оценки; EN: preserve undefined scores.
print('Parameters:',sum(p.numel() for p in model.parameters()),'loss:',history[0],history[-1],'test IoU:',np.nanmean(scores))  # RU: выводим измерения; EN: report measurements.
fig,axes = plt.subplots(1,3,figsize=(12,4))  # RU: сравниваем изображение, цель и результат; EN: compare image, target, and output.
name = PARTS['test'][0]  # RU: фиксируем показанный пример; EN: fix the displayed example.
axes[0].imshow(images[name]); axes[1].imshow(trimaps[name]==1); axes[2].imshow(cv2.resize(probabilities[0],(128,128))>=.5)  # RU: показываем настоящий неотобранный результат; EN: display an actual unfiltered result.
plt.show()  # RU: выводим диагностический рисунок; EN: render the diagnostic figure.
